# `Semana 9`: Threading

## 1. Threading

Para ejecutar varias cosas al mismo tiempo el sistema operativo usa **procesos**: un proceso es un programa en ejecución con su propio espacio de memoria.
- **Paralelismo**: varias tareas se ejecutan al mismo tiempo, en distintos núcleos/procesadores.
- **Concurrencia**: varias tareas se van intercalando en un mismo núcleo; los cambios de contexto son tan rápidos que aparentan simultaneidad, aunque en un instante dado solo una se está ejecutando.

Un *thread* es una unidad de ejecución dentro de un proceso. Todo proceso nace con al menos un *thread* : el *thread principal* (`MainThread`), pero se pueden crear más para que el programa ejecute varias partes de su código de forma concurrente. Cada *thread* mantiene su propio registro de en qué línea va y sus variables locales, pero comparte la memoria del proceso con los demás *threads*.

En Python, el módulo `threading` representa cada *thread* como una instancia de la clase `Thread`. Hay dos formas de crear uno:

1. Con una función `target`: se le entrega la función que el *thread* debe ejecutar (y sus argumentos vía `args`/`kwargs`). Crear la instancia **no** ejecuta nada; hay que llamar `.start()` para que comience. Un `Thread` creado así es de un solo uso: una vez ejecutado, no se puede reiniciar.
2. Heredando de `Thread`: se sobreescribe el método `run()` con las instrucciones que el *thread* debe ejecutar. `start()` llama internamente a `run()`.

`threading.current_thread()` retorna el *thread* que está ejecutando el código actual, y con ello acceder a sus atributos (como `.name`).

In [ ]:
import threading
from time import sleep

# Revisamos el thread actual
print(f"Este es el {threading.current_thread().name}")

# Definimos la función que ejecutará el thread
def thread_1(persona : str) -> None:
    segundos = 0
    thread_actual = threading.current_thread().name
    for i in range(1, 6):
        print(f"[{persona}] Estoy en el {thread_actual} y "
              + ("ha" if segundos == 1 else "han")
              + f" pasado {segundos} " 
              + ("segundo." if segundos == 1 else "segundos."))
        
        segundos += 1
        sleep(1)
        
hilo_1 = threading.Thread(name="Thread 1",target=thread_1, args=("Boney M",)) # Importante la ",", para señalizar una tupla de un elemento.

hilo_1.start()

Este es el MainThread
[Boney M] Estoy en el Thread 1 y han pasado 0 segundos.


[Boney M] Estoy en el Thread 1 y ha pasado 1 segundo.
[Boney M] Estoy en el Thread 1 y han pasado 2 segundos.
[Boney M] Estoy en el Thread 1 y han pasado 3 segundos.
[Boney M] Estoy en el Thread 1 y han pasado 4 segundos.


In [22]:
# Crearemos ahora una subclase de Thread

class MiThread(threading.Thread):
    def __init__(self, nombre : str, func_nombre : str) -> None:
        super().__init__(name=nombre)
        self.func_nombre = func_nombre
    
    # Hacemos overriding de run() 
    def run(self):
        segundos = 0
        thread_actual = threading.current_thread().name
        for i in range(1, 10):
            print(f"[{self.func_nombre}] Actualmente en {thread_actual} y tiempo "
                  + ("segundo" if segundos == 1 else "segundos")
                  + f" {segundos}." )
            
            segundos += 0.5
            sleep(0.5)
            
hilo_1 = threading.Thread(name="Thread 1",target=thread_1, args=("Boney M",))       
hilo_2 = MiThread("Thread 2", "Tommy Richman")

hilo_1.start()
hilo_2.start()


[Boney M] Estoy en el Thread 1 y han pasado 0 segundos.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 0.


[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 0.5.
[Boney M] Estoy en el Thread 1 y ha pasado 1 segundo.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundo 1.0.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 1.5.
[Boney M] Estoy en el Thread 1 y han pasado 2 segundos.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 2.0.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 2.5.
[Boney M] Estoy en el Thread 1 y han pasado 3 segundos.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 3.0.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 3.5.
[Boney M] Estoy en el Thread 1 y han pasado 4 segundos.
[Tommy Richman] Actualmente en Thread 2 y tiempo segundos 4.0.


A veces el programa principal (u otro *thread*) necesita **esperar** a que uno o más *threads* terminen antes de continuar. Para eso se usa `mi_thread.join()`, que bloquea a quien lo llama hasta que `mi_thread` termine.

> **Nota**: Si se hace `start()` y `join()` inmediatamente uno tras otro para cada *thread* (en vez de primero iniciar todos y luego esperar a todos), en la práctica no hay concurrencia real: cada *thread* se ejecuta completamente antes de iniciar el siguiente.

Con `mi_thread.is_alive()` se puede saber si un *thread* todavía está en ejecución.

Un `daemon thread` no impide que el programa principal termine, aunque siga corriendo. Si el programa principal termina, los *daemon threads* son interrumpidos abruptamente. Se configuran con el parámetro **daemon=True** al crear el Thread, o asignando **mi_thread.daemon = True** antes de llamar start() (después de start(), cambiarlo lanza `RuntimeError`).

In [21]:
hilo_1 = threading.Thread(name="Thread 1",target=thread_1, args=("Boney M",))

hilo_1.start()

main_thread = threading.current_thread()

# Hagamos que el hilo principal cuenta hasta 8 segundos
# y que, en el segundo 2, se pause para dar paso a hilo_1

for i in range(1, 9):
    print(f"[{main_thread.name}] Actualmente en <= {i} segundo(s)")
    
    if i == 2:
        print(f"[{main_thread.name}] Esperando a que {hilo_1.name} termine")
        
        # MainThread espera a que hilo_1 termine.
        # Si se le coloca, por otro lado,
        # hilo_1.join(timeout=10), esperará
        # solo 10 segundos a que termine, y sino,
        # MainThread seguirá con su ejecución.
        
        hilo_1.join()
        
        print("[Boney M] Finalizando...")
    sleep(1)

# Revisamos si hilo_1 sigue ejecutándose
print(">>> print(hilo_1.is_alive())")
print(hilo_1.is_alive())

[Boney M] Estoy en el Thread 1 y han pasado 0 segundos.
[MainThread] Actualmente en <= 1 segundo(s)
[Boney M] Estoy en el Thread 1 y ha pasado 1 segundo.
[MainThread] Actualmente en <= 2 segundo(s)
[MainThread] Esperando a que Thread 1 termine
[Boney M] Estoy en el Thread 1 y han pasado 2 segundos.
[Boney M] Estoy en el Thread 1 y han pasado 3 segundos.
[Boney M] Estoy en el Thread 1 y han pasado 4 segundos.
[Boney M] Finalizando...
[MainThread] Actualmente en <= 3 segundo(s)
[MainThread] Actualmente en <= 4 segundo(s)
[MainThread] Actualmente en <= 5 segundo(s)
[MainThread] Actualmente en <= 6 segundo(s)
[MainThread] Actualmente en <= 7 segundo(s)
[MainThread] Actualmente en <= 8 segundo(s)
>>> print(hilo_1.is_alive())
False


> **Nota**: El efecto del print con saltos de línea "raros", se debe a la misma concurrencia de los threads.

In [59]:
# Creamos un Daemon Thread

def mi_thread() -> None:
    actual = threading.current_thread()
    for i in range(3):
        print(f"[{actual.name}] Trabajando...")
        sleep(1)


# Creamos un daemon y uno normal
daemon = threading.Thread(target=mi_thread, name="Daemon", daemon=True)
normal = threading.Thread(target=mi_thread, name="Normal", daemon=False)

daemon.start()
normal.start()

sleep(1.5)

print("\n--- Comparamos ---")
print(f"> {daemon.name} -> is_alive() = {daemon.is_alive()}")
print(f"> {normal.name} -> is_alive() = {normal.is_alive()}")
print()

[Daemon] Trabajando...
[Normal] Trabajando...
[Daemon] Trabajando...
[Normal] Trabajando...

--- Comparamos ---
> Daemon -> is_alive() = True
> Normal -> is_alive() = True



[Normal] Trabajando...
[Daemon] Trabajando...


La diferencia real entre el daemon y el normal es que, al cerrar el proceso, Python espera a los hilos no-daemon pero apaga a los daemon. En Jupyter, como el kernel nunca muere al terminar una celda, ambos hilos corren idénticos hasta acabar su bucle, por lo que en ambos se ve que están "vivos" (aunque en un archivo.py el normal se seguiría ejecutando mientras el daemon no)

`Timer` (**threading.Timer(segundos, funcion, args=..., kwargs=...)**) es una subclase de `Thread` que espera el tiempo indicado y luego ejecuta la función. Al igual que cualquier `Thread`, se debe llamar `.start()`. Además, un `Timer` se puede cancelar con `.cancel()` mientras está esperando.

In [74]:
# Podemos crear un Timer de dos formas.

# Opción 1: Crearlo desde la misma clase
# Timer de threading.

# Opción 2: Haciendo overriding de
# run() en una subclase de Thread


# Usaremos la primera por comodidad

# El thread espera 3 segundos antes de ejecutarse
# Acá target se cambia por function
mi_timer = threading.Timer(interval=3, function=mi_thread)

# Para cambiar el nombre, se le tiene que hacer después de instanciarlo
mi_timer.name = "Timer 1"

print("Esperamos 3 segundos y...")
mi_timer.start()

Esperamos 3 segundos y...


[Timer 1] Trabajando...
[Timer 1] Trabajando...
[Timer 1] Trabajando...


In [77]:
mi_timer_cancelado = threading.Timer(interval=3, function=mi_thread)

mi_timer_cancelado.name = "Timer 2"

print("Esperamos 3 segundos y...")
mi_timer_cancelado.start()

sleep(2)

# Cancelamos el timer después de 2 segundos.
mi_timer_cancelado.cancel()
print("Ups...")
print("Se canceló el timer 💀")

Esperamos 3 segundos y...
Ups...
Se canceló el timer 💀


## 2. Concurrencia y sus Consecuencias

Cuando varios *threads* acceden y modifican **el mismo recurso** (una variable, un archivo, etc.) al mismo tiempo, pueden producirse resultados inesperados, porque:
- Las operaciones de un *thread* pueden ser interrumpidas en cualquier momento para dar paso a otro.
- No hay ninguna garantía sobre el orden en que se intercalan las instrucciones de distintos *threads* (lo decide el sistema operativo).

A la porción de código que **no debe** ser ejecutada por más de un *thread* a la vez se le llama **sección crítica**. Para que una operación sea segura entre *threads*, debe ser **atómica** dentro de esa sección.

Un **Lock** (de `threading.Lock`) nos sirve para sincronizar threads: puede estar bloqueado o desbloqueado. Un *thread* debe llamar `lock.acquire()` antes de entrar a la sección crítica, y `lock.release()` al salir. Mientras un *thread* mantiene el *lock* adquirido, cualquier otro *thread* que llame `acquire()` sobre el **mismo** *lock* deberá esperar.

**IMPORTANTE**: El `Lock` usado por todos los *threads* debe ser **el mismo objeto**. Si cada *thread* crea su propio `Lock()`, no hay ninguna protección real.

> **Nota**: También se puede usar un *lock* como *context manager* con `with lock:`, que llama automáticamente a `acquire()` y `release()`. Es recomendado hacerlo casi SIEMPRE de esta manera.

In [ ]:
# Crearemos una subclase de Thread con una variable de clase que sea un Lock.
# De esta manera, podremos tener un lock para todos los threads de manera
# más cómoda.

class Persona(threading.Thread):
    
    # Creamos el lock para toda la clase
    lock_global = threading.Lock()
    
    def __init__(self, nombre) -> None:
        super().__init__(name=nombre)
        
    def run(self) -> None:
        thread_actual = threading.current_thread()
        
        print(f"[{thread_actual.name}] Voy a entrar, a ver si está ocupado...")
        sleep(0.5)
        
        
        # Aquí es donde se verifica si el lock está ocupado o no.
        # Si está ocupado, todos los hilos que
        # quieran entrar al mismo tiempo, van a
        # congelarse y esperar que el lock se libere.
        
        with Persona.lock_global:
            # El Thread entra a la sección crítica.
            
            print(f"\n[{thread_actual.name}] Ocupando el baño...")
            sleep(2)
            print(f"[{thread_actual.name}] Saliendo del baño...")
            
        # Se libera el lock, y pasa al azar
        # cualquier thread congelado.
        
persona_1 = Persona("Pepito")
persona_2 = Persona("Juan")
persona_3 = Persona("Cristian")

persona_1.start()
sleep(0.2)
persona_2.start()
sleep(0.2)
persona_3.start()

[Pepito] Voy a entrar, a ver si está ocupado...
[Juan] Voy a entrar, a ver si está ocupado...
[Cristian] Voy a entrar, a ver si está ocupado...



[Pepito] Ocupando el baño...
[Pepito] Saliendo del baño...

[Juan] Ocupando el baño...
[Juan] Saliendo del baño...

[Cristian] Ocupando el baño...
[Cristian] Saliendo del baño...


Un **Event** (`threading.Event`) permite que un *thread* espere a que ocurra algo **durante** la vida de otro *thread* (a diferencia de `join()`, que solo espera a que un *thread* **termine** por completo).

Métodos principales:
- `event.set()`: activa la señal.
- `event.clear()`: desactiva la señal.
- `event.wait()`: bloquea al *thread* actual hasta que la señal esté activa.
- `event.is_set()`: retorna si la señal está activa o no, sin bloquear.

> **Nota**: Con **wait()**, todos los threads que están esperando al evento, **NO** funcionan como una cola: se activan todos al mismo tiempo.

In [16]:
# Simulemos una cola para ir al baño
# utilizando el ejemplo anterior

class Persona(threading.Thread):
    
    # Creamos las variable de clase
    banio_libre = threading.Event()
    banio_libre.set()


    def __init__(self, nombre) -> None:
        super().__init__(name=nombre)
        
    def run(self) -> None:
        thread_actual = threading.current_thread()
        
        print(f"[{thread_actual.name}] Voy a entrar, a ver si está ocupado...")
        sleep(0.5)
        
        # A las variables de clase se les puede acceder
        # tanto con self.variable o NombreClase.variable
        
        print(f"[{thread_actual.name}] Está el baño libre: {self.banio_libre.is_set()} ")
        
        if not self.banio_libre.is_set():
            # Congelamos al thread actual 
            # si el baño no está libre.
            self.banio_libre.wait()

        # Se ocupa el baño
        self.banio_libre.clear()
        
        print(f"\n[{thread_actual.name}] Ocupando el baño...\n")
        sleep(3)
        print(f"[{thread_actual.name}] Saliendo del baño...")
        
        # Se desocupa el baño
        self.banio_libre.set()
        
        
        
persona_1 = Persona("Pepito")
persona_2 = Persona("Juan")
persona_3 = Persona("Cristian")

persona_1.start()
sleep(1)
persona_2.start()
sleep(1)
persona_3.start()

[Pepito] Voy a entrar, a ver si está ocupado...
[Pepito] Está el baño libre: True 

[Pepito] Ocupando el baño...

[Juan] Voy a entrar, a ver si está ocupado...
[Juan] Está el baño libre: False 
[Cristian] Voy a entrar, a ver si está ocupado...


[Cristian] Está el baño libre: False 
[Pepito] Saliendo del baño...

[Juan] Ocupando el baño...


[Cristian] Ocupando el baño...

[Cristian] Saliendo del baño...[Juan] Saliendo del baño...



Al salir de Pepito del baño, todos los threads que lo estaban esperando, entran al mismo tiempo, por lo que **wait()** activa a todos simultáneamente. 

Revisemos mejor el sistema para la cola con el módulo `Queue`:
- `queue.put(item)`: agrega un elemento al final.
- `queue.get()`: retira un elemento del inicio, **bloqueando** al *thread* si la cola está vacía hasta que haya algo disponible.
- `task_done()`: cada vez que se procese un ítem de la cola.
- `join()`: El *thread* que lo llama se queda congelado hasta que la cola se vacíe.

> **queue.Queue**: Diseñada para multithreading. Es segura para hilos y congela el hilo de ejecución de forma automática si la cola está vacía hasta que reciba datos.
>
> **collections.deque**: Diseñada para rendimiento algorítmico, útil para un sólo hilo.


In [ ]:
from __future__ import annotations
import queue

# El sistema del pestillo funcionará así:
# Cada Thread de Persona se agregará a la cola del pestillo, y se quedará esperando con wait().
# Luego, el pestillo junto a get() obtendrá a la persona, esperará a que ésta se desocupe,
# y continuará a la siguiente en la cola, repitiéndose el ciclo.

class Pestillo(threading.Thread):
    def __init__(self, nombre) -> None:
        super().__init__(name=nombre)
        
        self.cola = queue.Queue()
        
    def esperar(self, persona : Persona):
        # Agregamos a la cola
        self.cola.put(persona)
        
    def run(self):
        while True:
            print("="*50)
            print(f"[{threading.current_thread().name}] Procesando personas...")
            print("="*50)
            
            # Obtenemos a la primera persona en la cola
            # junto a sus atributos
            persona : Persona = self.cola.get()
            banio_libre, desocupada = (persona.banio_libre, persona.desocupada)
            
            # La persona se descongela y puede entrar
            banio_libre.set()
            
            # Esperamos a que la persona se desocupe 
            # para avanzar
            sleep(0.1)
            desocupada.wait()
            
            self.cola.task_done()
    
pestillo = Pestillo("Pestillo")        
        
class Persona(threading.Thread):
    def __init__(self, nombre) -> None:
        super().__init__(name=nombre)
        self.banio_libre = threading.Event()
        self.desocupada = threading.Event()
        self.desocupada.set()
        
    def run(self) -> None:
        thread_actual = threading.current_thread()
        
        print(f"[{thread_actual.name}] Voy a entrar, a ver si está ocupado...")
        sleep(0.5)
        
        # Agregamos a la persona a la cola
        pestillo.esperar(self)
        
        # La persona está ocupada
        self.desocupada.clear()
        # La persona espera a que esté el baño libre
        self.banio_libre.wait()
        
        # Como pasó el wait, el baño está libre.
        
        print(f"\n[{thread_actual.name}] Ocupando el baño...\n")
        sleep(3)
        print(f"[{thread_actual.name}] Saliendo del baño...")

        # La desocupamos y le mandamos la señal
        # al pestillo
        self.desocupada.set()
        
persona_1 = Persona("Pepito")
persona_2 = Persona("Juan")
persona_3 = Persona("Cristian")

pestillo.start()

sleep(1)
persona_1.start()
sleep(1)
persona_2.start()
sleep(1)
persona_3.start()
        

[Pestillo] Procesando personas...
[Pepito] Voy a entrar, a ver si está ocupado...

[Pepito] Ocupando el baño...

[Juan] Voy a entrar, a ver si está ocupado...
[Cristian] Voy a entrar, a ver si está ocupado...


[Pepito] Saliendo del baño...
[Pestillo] Procesando personas...

[Juan] Ocupando el baño...

[Juan] Saliendo del baño...
[Pestillo] Procesando personas...

[Cristian] Ocupando el baño...

[Cristian] Saliendo del baño...
[Pestillo] Procesando personas...


### *Deadlocks*

Un ***deadlock*** (o interbloqueo) ocurre cuando dos o más *threads* se quedan esperándose mutuamente para siempre, sin que ninguno pueda avanzar.

In [34]:
# Creamos dos candados
lock_1 = threading.Lock()
lock_2 = threading.Lock()

def hilo_A():
    with lock_1:
        print("[Hilo A] Adquirió Lock 1. Esperando Lock 2...")
        sleep(0.1)  # Le da tiempo al Hilo B de arrancar y tomar el Lock 2
        with lock_2:     # Se congela aquí porque el Hilo B lo tiene ocupado
            print("[Hilo A] Esto nunca se ejecutará")

def hilo_B():
    with lock_2:
        print("[Hilo B] Adquirió Lock 2. Esperando Lock 1...")
        sleep(0.1)  # Le da tiempo al Hilo A de tomar el Lock 1
        with lock_1:     # Se congela aquí porque el Hilo A lo tiene ocupado
            print("[Hilo B] Esto nunca se ejecutará")

# Ejecutamos ambos hilos en paralelo
t1 = threading.Thread(target=hilo_A)
t2 = threading.Thread(target=hilo_B)

t1.start()
t2.start()


[Hilo A] Adquirió Lock 1. Esperando Lock 2...
[Hilo B] Adquirió Lock 2. Esperando Lock 1...


In [ ]:
# Ocurriendo que,

# hilo_A obtiene lock_1, hilo_B obtiene lock_2
# hilo_A quiere obtener lock_2, pero lo tiene hilo_B, así que lo espera y se congela.
# hilo_B quiere obtener lock_1, pero lo tiene hilo_A, así que lo espera y se congela.

# Ambos hilos se quedan congelados


# Guía rápida para aplicaciones de `threading`

| Patrón / Herramienta | Caso de uso | Esencial|
| :--- | :--- | :--- |
| **Lock de clase** | Recurso compartido por múltiples hilos de una clase | `lock = threading.Lock()` (variable de clase) |
| **`queue.Queue` (NO deque)** | Prod/Consumidor| `put()`, `get()`, `task_done()`, `join()` |
| **Multi-Queue** | Notificaciones / bandejas por usuario | una `queue.Queue()` por cada instancia de hilo |
| **`Event` + Daemon** | Sincronización de turnos/señales + trabajo de fondo| `ev.wait()`, `ev.set()`, `ev.clear()`, `daemon=True` |